# Notebook 5 — NLP (Natural Language Processing)
### BNP Paribas — Senior AI Engineer

---
## Sommaire
1. [Tokenisation](#tokenization)
2. [Représentations textuelles (Bag of Words, TF-IDF)](#bow)
3. [Word Embeddings (Word2Vec, GloVe, FastText)](#embeddings)
4. [Modèles de séquences (RNN, Seq2Seq, attention classique)](#seq)
5. [Attention & Self-Attention](#attention)
6. [NER — Named Entity Recognition](#ner)
7. [Language Modeling](#lm)
8. [Questions d’entretien](#questions)


---
## 1. Tokenisation <a name='tokenization'></a>

### Niveaux de tokenisation
| Niveau | Exemple | Avantages | Inconvénients |
|---|---|---|---|
| Caractère | h, e, l, l, o | Vocab petit, gère l'inconnu | Sequences tres longues |
| Mot | hello, world | Intuitif | Vocab immense, OOV |
| Sous-mot (BPE) | hel, lo, wor, ld | Equilibre vocab/longueur | Plus complexe |
| Phrase/Chunk | NER, POS | Semantique riche | Tres long |

### Algorithmes BPE & WordPiece
- **BPE (Byte Pair Encoding)** : fusion iterative des paires de tokens les plus frequentes
  - Utilise par : GPT-2, GPT-3, GPT-4, RoBERTa
- **WordPiece** : similaire a BPE mais maximise la vraisemblance du LM
  - Utilise par : BERT, DistilBERT
- **SentencePiece** : travaille directement sur les bytes, language-agnostic
  - Utilise par : T5, LLaMA, Mistral

### Vocabulaire typique
| Modele | Taille vocab | Algo |
|---|---|---|
| BERT | 30,522 | WordPiece |
| GPT-2 | 50,257 | BPE |
| LLaMA 2 | 32,000 | SentencePiece |
| GPT-4 (cl100k) | 100,277 | tiktoken BPE |


In [ ]:
import re
from collections import Counter, defaultdict

# ============================================================
# TOKENISATION DE BASE
# ============================================================
text = "The quick brown fox jumps over the lazy dog. Don't stop learning!"

# 1. Word tokenization (naive)
def word_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

# 2. Character tokenization
def char_tokenize(text):
    return list(text.lower())

# 3. Regex tokenization
def regex_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)

print("=== Tokenisation de base ===")
print(f"Texte: {text[:50]}...")
print(f"\nWord tokens ({len(word_tokenize(text))}) : {word_tokenize(text)[:10]}...")
print(f"Char tokens ({len(char_tokenize(text))}) : {char_tokenize(text)[:15]}...")
print(f"Regex tokens ({len(regex_tokenize(text))}) : {regex_tokenize(text)[:12]}...")

# ============================================================
# BPE (Byte Pair Encoding) -- from scratch
# ============================================================
class BPETokenizer:
    """Implementation simplifiee de BPE"""

    def __init__(self, vocab_size=100):
        self.vocab_size = vocab_size
        self.merges = {}
        self.vocab = set()

    def _get_pairs(self, vocab):
        """Compte toutes les paires adjacentes"""
        pairs = Counter()
        for word, freq in vocab.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs

    def _merge_vocab(self, pair, vocab):
        """Fusionne une paire dans tout le vocabulaire"""
        bigram = re.escape(" ".join(pair))
        replacement = "".join(pair)
        new_vocab = {}
        for word in vocab:
            new_word = re.sub(bigram, replacement, word)
            new_vocab[new_word] = vocab[word]
        return new_vocab

    def fit(self, corpus):
        """Apprendre les fusions BPE"""
        # Init: chaque mot = sequence de caracteres + </w> (fin de mot)
        vocab = Counter()
        for text in corpus:
            for word in text.lower().split():
                vocab[" ".join(list(word)) + " </w>"] += 1

        self.vocab = set()
        for word in vocab:
            for char in word.split():
                self.vocab.add(char)

        n_merges = self.vocab_size - len(self.vocab)
        for i in range(n_merges):
            pairs = self._get_pairs(vocab)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            vocab = self._merge_vocab(best, vocab)
            self.merges[best] = "".join(best)
            self.vocab.add("".join(best))

        return self

    def tokenize(self, word):
        """Tokenise un mot selon les fusions apprises"""
        word_chars = " ".join(list(word)) + " </w>"
        for pair, merged in self.merges.items():
            bigram = re.escape(" ".join(pair))
            word_chars = re.sub(bigram, merged, word_chars)
        return word_chars.split()

corpus = [
    "low lower lowest",
    "newer new newest",
    "wider wide width",
    "low low low newer newer",
]
bpe = BPETokenizer(vocab_size=30).fit(corpus)

print("\n=== BPE Tokenizer ===")
print(f"Nombre de fusions apprises : {len(bpe.merges)}")
print(f"Taille du vocabulaire      : {len(bpe.vocab)}")
print(f"\nPremiers merges: {list(bpe.merges.items())[:5]}")
print()
for word in ["low", "lower", "lowest", "newest", "unknown"]:
    tokens = bpe.tokenize(word)
    print(f"  '{word}' => {tokens}")


In [ ]:
# ============================================================
# TOKENISATION PRATIQUE AVEC HUGGINGFACE
# ============================================================
print("=== Tokenisation HuggingFace (simulation sans import) ===")

# Ce code illustre ce que fait HuggingFace en pratique
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# tokens = tokenizer("Hello, how are you?", return_tensors="pt")

# Simulation manuelle du comportement WordPiece de BERT
bert_vocab_sample = {
    "[CLS]": 101, "[SEP]": 102, "[PAD]": 0, "[UNK]": 100, "[MASK]": 103,
    "hello": 7592, ",": 1010, "how": 2129, "are": 2024, "you": 2017,
    "bank": 2924, "##ing": 6620, "##rupt": 16062, "##cy": 5766,
    "play": 2377, "##ing": 6620, "##er": 2121,
    "un": 4895, "##want": 16893, "##ed": 2098
}

def wordpiece_tokenize(text, vocab):
    """Simulation de WordPiece BERT"""
    tokens = ["[CLS]"]
    for word in text.lower().split():
        if word in vocab:
            tokens.append(word)
        else:
            # Decomposition en sous-mots
            remaining = word
            word_tokens = []
            while remaining:
                found = False
                for end in range(len(remaining), 0, -1):
                    subword = remaining[:end]
                    if word_tokens:
                        subword = "##" + subword
                    if subword in vocab:
                        word_tokens.append(subword)
                        remaining = remaining[end:]
                        found = True
                        break
                if not found:
                    word_tokens.append("[UNK]")
                    break
            tokens.extend(word_tokens)
    tokens.append("[SEP]")
    return tokens

texts = [
    "hello how are you",
    "banking bankruptcy",
    "playing player unwanted"
]
for text in texts:
    tokens = wordpiece_tokenize(text, bert_vocab_sample)
    ids    = [bert_vocab_sample.get(t, 100) for t in tokens]
    print(f"\n  Input   : {text}")
    print(f"  Tokens  : {tokens}")
    print(f"  IDs     : {ids}")

print("\n=== Proprietes cles des tokeniseurs modernes ===")
print("  1. Vocabulaire ferme de taille fixe (ex: 32k-100k tokens)")
print("  2. Gestion de l OOV: decomposition en sous-mots connus")
print("  3. Tokens speciaux: [CLS], [SEP], [PAD], [MASK], <s>, </s>")
print("  4. Encodage des positions (separee ou integree)")
print("  5. Attention_mask: 1 si token reel, 0 si padding")


---
## 2. Représentations textuelles : BoW & TF-IDF <a name='bow'></a>

### Bag of Words
- Vecteur de frequences de mots, ignore l'ordre
- Sparse, grande dimension

### TF-IDF
```
TF(t, d)  = count(t in d) / len(d)
IDF(t)    = log(N / df(t))           # df = nb de docs contenant t
TF-IDF    = TF * IDF
```
- Penalise les mots tres frequents (le, la, de)
- Valorise les mots discriminants

### Limites
- Pas de semantique (roi et monarque sont independants)
- Sparse, pas adapte aux reseaux de neurones
- Remplace par les **embeddings denses** dans les modeles modernes


In [ ]:
import numpy as np
from collections import Counter
import math

# ============================================================
# TF-IDF FROM SCRATCH
# ============================================================
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog played",
    "machine learning is great for NLP tasks",
    "deep learning models learn representations",
    "NLP tasks include classification and generation",
]

def build_tfidf(corpus):
    """Calcul TF-IDF from scratch"""
    # Tokenisation
    tokenized = [doc.lower().split() for doc in corpus]
    N = len(tokenized)

    # Vocabulaire
    all_words = sorted(set(w for doc in tokenized for w in doc))
    word2idx = {w: i for i, w in enumerate(all_words)}
    V = len(all_words)

    # TF: freq relative dans chaque document
    tf = np.zeros((N, V))
    for i, doc in enumerate(tokenized):
        counts = Counter(doc)
        for word, count in counts.items():
            tf[i, word2idx[word]] = count / len(doc)

    # DF: nombre de documents contenant chaque mot
    df = np.zeros(V)
    for i, doc in enumerate(tokenized):
        for word in set(doc):
            df[word2idx[word]] += 1

    # IDF: log(N/df) -- lissage +1 pour eviter division par zero
    idf = np.log((N + 1) / (df + 1)) + 1.0  # smooth IDF

    # TF-IDF
    tfidf = tf * idf

    # L2-normalisation (cosine similarity ensuite)
    norms = np.linalg.norm(tfidf, axis=1, keepdims=True)
    tfidf_norm = np.divide(tfidf, norms, where=norms > 0)

    return tfidf_norm, all_words, idf

tfidf_matrix, vocab, idf_scores = build_tfidf(corpus)

print("=== TF-IDF ===")
print(f"Corpus: {len(corpus)} documents, vocabulaire: {len(vocab)} mots")
print(f"Matrice TF-IDF: {tfidf_matrix.shape}")

# Top mots discriminants (IDF eleve)
top_idf = sorted(zip(vocab, idf_scores), key=lambda x: -x[1])[:8]
print(f"\nMots les plus discriminants (IDF eleve):")
for word, score in top_idf:
    print(f"  {word:20s}: IDF={score:.4f}")

# Similarite cosinus entre documents
def cosine_sim(v1, v2):
    return np.dot(v1, v2)  # deja normalises en L2

print(f"\nSimilarite cosinus:")
print(f"  Doc0 vs Doc1 (meme sujet cat/dog): {cosine_sim(tfidf_matrix[0], tfidf_matrix[1]):.4f}")
print(f"  Doc0 vs Doc3 (sujets differents) : {cosine_sim(tfidf_matrix[0], tfidf_matrix[3]):.4f}")
print(f"  Doc3 vs Doc4 (meme sujet ML)     : {cosine_sim(tfidf_matrix[3], tfidf_matrix[4]):.4f}")


---
## 3. Word Embeddings <a name='embeddings'></a>

### Word2Vec
Apprend des vecteurs denses en predisant :
- **CBOW** (Continuous Bag of Words) : predict le mot central a partir du contexte
- **Skip-gram** : predict le contexte a partir du mot central
- **Negative sampling** : optimisation avec des exemples negatifs

### Proprietes emergentes
```
vec("king") - vec("man") + vec("woman") ~ vec("queen")
vec("Paris") - vec("France") + vec("Germany") ~ vec("Berlin")
```

### Comparaison des embeddings
| Methode | Type | Contexte | Params |
|---|---|---|---|
| Word2Vec | Statique | Non | Leger |
| GloVe | Statique (cooccurrence globale) | Non | Leger |
| FastText | Statique + n-grams | Non | Leger, gere OOV |
| ELMo | Contextuel (LSTM bidir) | Oui | Moyen |
| BERT | Contextuel (Transformer) | Oui | Lourd |

> **Embeddings contextuels** : "bank" aura un vecteur different dans "river bank" vs "bank account"


In [ ]:
import numpy as np
from collections import defaultdict

# ============================================================
# SKIP-GRAM WITH NEGATIVE SAMPLING -- from scratch
# ============================================================
class Word2VecSkipGram:
    """Implementation simplifiee de Skip-gram"""

    def __init__(self, vocab_size, embed_dim=10, lr=0.025):
        self.V = vocab_size
        self.d = embed_dim
        self.lr = lr
        # W_in: embedding des mots cibles (lookup table)
        self.W_in  = np.random.randn(vocab_size, embed_dim) * 0.01
        # W_out: embedding du contexte (poids de sortie)
        self.W_out = np.random.randn(vocab_size, embed_dim) * 0.01

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def train_pair(self, center_idx, context_idx, neg_indices):
        """Un pas de gradient pour une paire (center, context) + negatifs"""
        v_c = self.W_in[center_idx]  # vecteur du mot central

        # Paire positive
        score_pos = np.dot(v_c, self.W_out[context_idx])
        sig_pos   = self.sigmoid(score_pos)
        loss      = -np.log(sig_pos + 1e-10)

        # Gradient pour le mot de contexte positif
        self.W_out[context_idx] -= self.lr * (sig_pos - 1) * v_c
        grad_center = (sig_pos - 1) * self.W_out[context_idx]

        # Paires negatives
        for neg_idx in neg_indices:
            score_neg = np.dot(v_c, self.W_out[neg_idx])
            sig_neg   = self.sigmoid(score_neg)
            loss     += -np.log(1 - sig_neg + 1e-10)
            self.W_out[neg_idx] -= self.lr * sig_neg * v_c
            grad_center         += sig_neg * self.W_out[neg_idx]

        self.W_in[center_idx] -= self.lr * grad_center
        return loss

    def get_embedding(self, idx):
        return self.W_in[idx]

    def most_similar(self, idx, top_k=5, idx2word=None):
        v = self.W_in[idx]
        sims = self.W_in @ v / (np.linalg.norm(self.W_in, axis=1) * np.linalg.norm(v) + 1e-10)
        sims[idx] = -1  # exclure le mot lui-meme
        top = np.argsort(sims)[::-1][:top_k]
        if idx2word:
            return [(idx2word[i], sims[i]) for i in top]
        return [(i, sims[i]) for i in top]

# Corpus simplifie
sentences = [
    "the king rules the kingdom",
    "the queen rules the kingdom",
    "the man is brave and strong",
    "the woman is brave and kind",
    "paris is the capital of france",
    "berlin is the capital of germany",
    "france and germany are countries",
    "the king and queen of france",
]

# Construction vocabulaire
all_words = [w for s in sentences for w in s.split()]
vocab = sorted(set(all_words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)
print(f"=== Skip-gram Word2Vec ===")
print(f"Vocabulaire: {V} mots: {vocab}")

# Generer les paires (center, context)
window = 2
pairs = []
for sentence in sentences:
    words = sentence.split()
    for i, center in enumerate(words):
        for j in range(max(0,i-window), min(len(words),i+window+1)):
            if i != j:
                pairs.append((word2idx[center], word2idx[words[j]]))

# Entrainement
np.random.seed(42)
model = Word2VecSkipGram(V, embed_dim=8, lr=0.1)
losses = []
for epoch in range(300):
    epoch_loss = 0
    for center, context in pairs:
        neg = np.random.choice(V, size=3, replace=False)
        neg = neg[neg != center][:2]
        epoch_loss += model.train_pair(center, context, neg)
    losses.append(epoch_loss / len(pairs))

print(f"\nLoss initiale : {losses[0]:.4f}")
print(f"Loss finale   : {losses[-1]:.4f}")


In [ ]:
import numpy as np

# (suite du Word2Vec entraine)
# ============================================================
# PROPRIETES DES EMBEDDINGS
# ============================================================
# On re-cree le modele rapidement pour la demo
sentences = [
    "the king rules the kingdom", "the queen rules the kingdom",
    "the man is brave and strong", "the woman is brave and kind",
    "paris is the capital of france", "berlin is the capital of germany",
    "france and germany are countries", "the king and queen of france",
]
all_words = [w for s in sentences for w in s.split()]
vocab = sorted(set(all_words))
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
V = len(vocab)

# Charger un embedding pre-entraine simule (avec les bonnes analogies)
np.random.seed(42)
embed_dim = 4
embeddings = np.random.randn(V, embed_dim) * 0.1

# Simuler les proprietes semantiques manuellement
# king ~ [1, 0, 1, 0], queen ~ [1, 0, 0, 1], man ~ [0, 1, 1, 0], woman ~ [0, 1, 0, 1]
# paris ~ [0.5, 0, 1, 0], berlin ~ [0.5, 0, 0, 1], france ~ [0, 0.5, 1, 0], germany ~ [0, 0.5, 0, 1]
semantic_vectors = {
    "king": [1.0, 0.0, 1.0, 0.0], "queen": [1.0, 0.0, 0.0, 1.0],
    "man":  [0.0, 1.0, 1.0, 0.0], "woman": [0.0, 1.0, 0.0, 1.0],
    "paris": [0.5, 0.0, 1.0, 0.0], "berlin": [0.5, 0.0, 0.0, 1.0],
    "france": [0.0, 0.5, 1.0, 0.0], "germany": [0.0, 0.5, 0.0, 1.0],
}
for word, vec in semantic_vectors.items():
    if word in word2idx:
        embeddings[word2idx[word]] = np.array(vec)

def cosine_sim(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10)

def analogy(a, b, c, embeddings, word2idx, idx2word, top_k=3):
    """a est a b ce que c est a ?"""
    v = embeddings[word2idx[b]] - embeddings[word2idx[a]] + embeddings[word2idx[c]]
    sims = np.array([cosine_sim(v, embeddings[i]) for i in range(len(embeddings))])
    # Exclure a, b, c
    for w in [a, b, c]:
        sims[word2idx[w]] = -1
    top = np.argsort(sims)[::-1][:top_k]
    return [(idx2word[i], sims[i]) for i in top]

print("=== Analogies vectorielles ===")
print("  Format: 'a' est a 'b' ce que 'c' est a ?")
tests = [("man", "king", "woman"), ("man", "woman", "king"), ("france", "paris", "germany")]
for a, b, c in tests:
    result = analogy(a, b, c, embeddings, word2idx, idx2word)
    print(f"  {a} -> {b} :: {c} -> {result[0][0]} (sim={result[0][1]:.3f})")

print("\n=== Mots les plus similaires ===")
for word in ["king", "paris", "france"]:
    idx = word2idx[word]
    sims = [(idx2word[i], cosine_sim(embeddings[idx], embeddings[i]))
            for i in range(V) if i != idx]
    sims.sort(key=lambda x: -x[1])
    print(f"  Similaires a '{word}': {sims[:3]}")

print("\n=== FastText : gestion des mots hors-vocabulaire ===")
print("  Word2Vec/GloVe: OOV -> [UNK]")
print("  FastText: decompose en n-grammes de caracteres")
def fasttext_ngrams(word, min_n=3, max_n=6):
    word = "<" + word + ">"
    ngrams = set()
    for n in range(min_n, min(max_n+1, len(word)+1)):
        for i in range(len(word)-n+1):
            ngrams.add(word[i:i+n])
    return ngrams

for word in ["cat", "cats", "catz", "unfamiliarword"]:
    ng = fasttext_ngrams(word)
    print(f"  '{word}': {sorted(ng)[:5]}...")


---
## 4. Modèles de séquences <a name='seq'></a>

### Seq2Seq (Encoder-Decoder)
- **Encoder** : lit la séquence source, produit un vecteur de contexte `c`
- **Decoder** : génère la séquence cible token par token conditionnellement a `c`
- Problème : bottleneck — tout compresser dans un vecteur de taille fixe

### Attention de Bahdanau (2015)
- Le decoder peut "regarder" tous les hidden states de l'encoder
- Score : `e_ij = a(s_{i-1}, h_j)` (alignement)
- Poids : `alpha_ij = softmax(e_ij)`
- Contexte : `c_i = sum_j alpha_ij * h_j`

### Teacher Forcing
- Pendant l'entrainement : fournir le vrai token precedent au decoder
- Avantage : convergence plus rapide
- Inconvenient : exposure bias (au test, le decoder voit ses propres predictions)

### Strategies de decodage
| Strategie | Description | Usage |
|---|---|---|
| Greedy | Argmax a chaque pas | Rapide, sous-optimal |
| Beam Search | k meilleures sequences | Standard en traduction |
| Top-k sampling | Sample parmi les k meilleurs | Texte creatif |
| Top-p (nucleus) | Sample jusqu a p de proba cumulative | GPT, creativite |
| Temperature | Divise les logits par T | Controle la distribution |


In [ ]:
import numpy as np

# ============================================================
# ATTENTION DE BAHDANAU -- from scratch
# ============================================================
class BahdanauAttention:
    """Attention additive (Bahdanau 2015)"""
    def __init__(self, encoder_hidden, decoder_hidden, attention_dim):
        # W_a: projette l etat du decoder
        self.W_a = np.random.randn(attention_dim, decoder_hidden) * 0.01
        # U_a: projette les etats de l encoder
        self.U_a = np.random.randn(attention_dim, encoder_hidden) * 0.01
        # v_a: vecteur de score
        self.v_a = np.random.randn(attention_dim) * 0.01

    def score(self, decoder_state, encoder_states):
        """
        decoder_state  : (decoder_hidden,)
        encoder_states : (T, encoder_hidden)
        Retourne scores : (T,)
        """
        # proj decoder : (attention_dim,)
        proj_dec = self.W_a @ decoder_state
        # proj encoder : (T, attention_dim)
        proj_enc = encoder_states @ self.U_a.T
        # energie : tanh(proj_dec + proj_enc) * v_a
        energy = np.tanh(proj_dec + proj_enc) @ self.v_a
        return energy

    def forward(self, decoder_state, encoder_states):
        """
        Retourne le vecteur de contexte et les poids d attention
        """
        energy = self.score(decoder_state, encoder_states)
        # Softmax -> poids d attention
        alpha = np.exp(energy - energy.max())
        alpha = alpha / alpha.sum()
        # Vecteur de contexte : somme ponderee des etats encoder
        context = (alpha[:, np.newaxis] * encoder_states).sum(axis=0)
        return context, alpha

# Simulation d un encoder-decoder avec attention
np.random.seed(42)
T_src = 6    # longueur de la sequence source
T_tgt = 4    # longueur de la sequence cible
enc_h = 16   # hidden size encoder
dec_h = 16   # hidden size decoder
attn_dim = 8

# Hidden states de l encoder (simules)
encoder_states = np.random.randn(T_src, enc_h)
decoder_state  = np.random.randn(dec_h)

attn = BahdanauAttention(enc_h, dec_h, attn_dim)
context, alpha = attn.forward(decoder_state, encoder_states)

print("=== Attention de Bahdanau ===")
print(f"Sequence source  : T={T_src}")
print(f"Etat decoder     : {dec_h}D")
print(f"Poids d attention: {alpha.round(4)}")
print(f"  => somme={alpha.sum():.4f}")
print(f"  => le decoder se 'concentre' sur les positions {np.argsort(alpha)[::-1][:2]}")
print(f"Vecteur contexte : shape={context.shape}")
print("\n  Interpretation: alpha[j] = importance du j-eme token source")
print("  pour generer le token courant de la cible")


In [ ]:
import numpy as np

# ============================================================
# STRATEGIES DE DECODAGE
# ============================================================
def softmax(x, temperature=1.0):
    x = x / temperature
    e = np.exp(x - x.max())
    return e / e.sum()

def greedy_decode(logits_seq):
    """Greedy: argmax a chaque pas"""
    return [np.argmax(logits) for logits in logits_seq]

def top_k_sample(logits, k=5, temperature=1.0, seed=None):
    """Sample parmi les k meilleures probabilites"""
    rng = np.random.RandomState(seed)
    probs = softmax(logits, temperature)
    top_k_idx = np.argsort(probs)[-k:]
    top_k_probs = probs[top_k_idx]
    top_k_probs /= top_k_probs.sum()
    return rng.choice(top_k_idx, p=top_k_probs)

def top_p_nucleus_sample(logits, p=0.9, temperature=1.0, seed=None):
    """Nucleus sampling: sample parmi les tokens dont la proba cum = p"""
    rng = np.random.RandomState(seed)
    probs = softmax(logits, temperature)
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cum_probs = np.cumsum(sorted_probs)
    # Garder les tokens jusqu a ce que la proba cumulee depasse p
    cutoff = np.searchsorted(cum_probs, p) + 1
    top_idx = sorted_idx[:cutoff]
    top_probs = probs[top_idx]
    top_probs /= top_probs.sum()
    return rng.choice(top_idx, p=top_probs)

# Simulation sur un vocabulaire de 10 tokens
np.random.seed(42)
logits = np.array([0.1, 2.5, 0.3, 1.8, 0.2, 0.5, 0.1, 3.0, 0.4, 0.8])
vocab = ["<pad>","le","chat","chien","mange","dort","vite","lentement","rouge","bleu"]

probs = softmax(logits)
print("=== Strategies de decodage ===")
print(f"Token le plus probable: '{vocab[np.argmax(probs)]}' ({probs.max():.4f})")
print()
print(f"Greedy (T=1.0)     : '{vocab[np.argmax(probs)]}'")

for k_val in [3, 5]:
    tok = top_k_sample(logits, k=k_val, temperature=1.0, seed=42)
    print(f"Top-k (k={k_val}, T=1.0)  : '{vocab[tok]}'")

for p_val in [0.9, 0.95]:
    tok = top_p_nucleus_sample(logits, p=p_val, temperature=1.0, seed=42)
    print(f"Top-p (p={p_val}, T=1.0): '{vocab[tok]}'")

print("\n=== Effet de la temperature ===")
for T in [0.1, 0.5, 1.0, 1.5, 2.0]:
    p = softmax(logits, temperature=T)
    entropy = -np.sum(p * np.log(p + 1e-10))
    top_token = vocab[np.argmax(p)]
    print(f"  T={T}: top='{top_token}', max_prob={p.max():.4f}, entropy={entropy:.4f}")
print("  => T<1: distribution piquee (deterministe). T>1: distribution plate (creative)")


---
## 5. Attention & Self-Attention <a name='attention'></a>

### Scaled Dot-Product Attention
```
Attention(Q, K, V) = softmax(Q * K^T / sqrt(d_k)) * V
```
- **Q** (Query) : ce que je cherche
- **K** (Key)   : ce que j'offre comme cle de recherche
- **V** (Value) : ce que je donne si trouve

### Pourquoi diviser par sqrt(d_k) ?
Pour eviter que le produit scalaire ne devienne trop grand quand d_k est grand,
ce qui ferait saturer le softmax et rendrait les gradients tres petits.

### Multi-Head Attention
```
MultiHead(Q,K,V) = Concat(head_1, ..., head_h) * W_O
head_i = Attention(Q*W_Qi, K*W_Ki, V*W_Vi)
```
- Chaque tete apprend des relations differentes
- h=8 ou h=12 typiquement (BERT base: h=12, d_model=768, d_k=64)

### Masking
- **Padding mask** : ignorer les tokens [PAD] dans l'attention
- **Causal mask** (look-ahead) : le token t ne peut voir que t-1, t-2... (decodeur autoregressive)


In [ ]:
import numpy as np

# ============================================================
# SCALED DOT-PRODUCT ATTENTION -- from scratch
# ============================================================
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q : (batch, heads, seq_q, d_k)
    K : (batch, heads, seq_k, d_k)
    V : (batch, heads, seq_k, d_v)
    mask : (batch, 1, seq_q, seq_k)  optionnel
    """
    d_k = Q.shape[-1]

    # Scores: (batch, heads, seq_q, seq_k)
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)

    # Masking (remplace par -inf avant softmax)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    # Softmax sur la derniere dimension (seq_k)
    scores_shifted = scores - scores.max(axis=-1, keepdims=True)
    weights = np.exp(scores_shifted)
    weights = weights / (weights.sum(axis=-1, keepdims=True) + 1e-10)

    # Output: (batch, heads, seq_q, d_v)
    output = weights @ V
    return output, weights

# Demo
np.random.seed(42)
batch, heads, seq_len, d_k, d_v = 2, 4, 6, 16, 16

Q = np.random.randn(batch, heads, seq_len, d_k)
K = np.random.randn(batch, heads, seq_len, d_k)
V = np.random.randn(batch, heads, seq_len, d_v)

# 1. Sans masque
output, weights = scaled_dot_product_attention(Q, K, V)
print("=== Scaled Dot-Product Attention ===")
print(f"Q/K/V shape  : {Q.shape}")
print(f"Output shape : {output.shape}")
print(f"Weights shape: {weights.shape}")
print(f"Weights sum  : {weights[0, 0, 0].sum():.4f}  (doit etre 1)")

# 2. Causal mask (autoregressive)
causal_mask = np.tril(np.ones((seq_len, seq_len)))[np.newaxis, np.newaxis, :, :]
output_masked, weights_masked = scaled_dot_product_attention(Q, K, V, causal_mask)
print(f"\nCausal mask (triangulaire inferieur):")
print(causal_mask[0, 0].astype(int))
print(f"\nWeights masques (token 0 ne voit que lui-meme):")
print(weights_masked[0, 0].round(3))


In [ ]:
import numpy as np

# ============================================================
# MULTI-HEAD ATTENTION -- from scratch
# ============================================================
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        self.h = num_heads
        self.d_k = d_model // num_heads
        self.d_model = d_model

        # Projections Q, K, V et output
        scale = np.sqrt(2.0 / d_model)
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale

    def split_heads(self, x):
        """(batch, seq, d_model) -> (batch, heads, seq, d_k)"""
        batch, seq, _ = x.shape
        x = x.reshape(batch, seq, self.h, self.d_k)
        return x.transpose(0, 2, 1, 3)

    def scaled_attention(self, Q, K, V, mask=None):
        d_k = Q.shape[-1]
        scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)
        if mask is not None:
            scores = np.where(mask == 0, -1e9, scores)
        scores -= scores.max(axis=-1, keepdims=True)
        w = np.exp(scores)
        w /= w.sum(axis=-1, keepdims=True) + 1e-10
        return w @ V, w

    def forward(self, x, mask=None):
        batch, seq, _ = x.shape

        Q = x @ self.W_Q.T
        K = x @ self.W_K.T
        V = x @ self.W_V.T

        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        out, weights = self.scaled_attention(Q, K, V, mask)

        # Concatener les tetes
        out = out.transpose(0, 2, 1, 3).reshape(batch, seq, -1)
        return out @ self.W_O.T, weights

np.random.seed(42)
d_model, num_heads, seq_len, batch = 64, 8, 10, 2
mha = MultiHeadAttention(d_model, num_heads)

x = np.random.randn(batch, seq_len, d_model)
output, weights = mha.forward(x)

print("=== Multi-Head Attention ===")
print(f"Input    : {x.shape}  (batch={batch}, seq={seq_len}, d_model={d_model})")
print(f"Output   : {output.shape}")
print(f"Weights  : {weights.shape}  (batch, heads, seq_q, seq_k)")

# Nb de params
n_params = 4 * d_model * d_model
print(f"\nNb parametres MHA : {n_params:,}  (4 matrices {d_model}x{d_model})")
print(f"  d_k = d_model / h = {d_model}/{num_heads} = {d_model//num_heads}")
print(f"  Chaque tete voit des projections dans R^{d_model//num_heads}")
print("  => Chaque tete apprend des patterns d attention differents")
print("     ex: tete 1 = syntaxe, tete 2 = coreference, tete 3 = semantique...")

# ============================================================
# POSITIONAL ENCODING
# ============================================================
def positional_encoding(seq_len, d_model):
    """Encodage positionnel sinusoidal (Attention is All You Need)"""
    PE = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, np.newaxis]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    PE[:, 0::2] = np.sin(pos * div)
    PE[:, 1::2] = np.cos(pos * div[:d_model//2])
    return PE

PE = positional_encoding(seq_len=20, d_model=64)
print(f"\n=== Positional Encoding ===")
print(f"Shape: {PE.shape}")
print(f"Valeurs pos 0: {PE[0, :6].round(3)}")
print(f"Valeurs pos 1: {PE[1, :6].round(3)}")
print("  => Permet au modele de differencier 'le chat mange le chien' de 'le chien mange le chat'")
print("  => Sin/cos: les positions proches ont des PE similaires (inductive bias de localite)")


---
## 6. NER — Named Entity Recognition <a name='ner'></a>

### Schemas de labeling
- **BIO** : B-TYPE (Beginning), I-TYPE (Inside), O (Outside)
- **BIOES** : + E (End), S (Single token)

Exemple :
```
Jean    -> B-PER
Dupont  -> I-PER
travaille -> O
chez    -> O
BNP     -> B-ORG
Paribas -> I-PER
a       -> O
Paris   -> B-LOC
```

### Architectures
| Architecture | Annee | Notes |
|---|---|---|
| CRF sur features manuelles | Pre-2015 | Interpretable |
| BiLSTM-CRF | 2015 | Standard pre-transformers |
| BERT + token classification | 2019 | State of art |
| BERT + CRF | 2019 | Meilleur que BERT seul |

### CRF (Conditional Random Field)
- Modelise les **dependances entre labels** adjacents
- Score d'une sequence : `score(y) = sum emit(y_t, x_t) + sum trans(y_{t-1}, y_t)`
- Garantit des transitions valides (ex: I-PER ne peut pas suivre B-ORG)


In [ ]:
import numpy as np
from collections import defaultdict

# ============================================================
# NER AVEC SCHEMA BIO -- evaluation
# ============================================================
def bio_to_spans(labels):
    """Convertit une sequence BIO en liste de spans (type, start, end)"""
    spans = []
    current = None
    start = None
    for i, label in enumerate(labels):
        if label.startswith("B-"):
            if current:
                spans.append((current, start, i))
            current = label[2:]
            start = i
        elif label.startswith("I-"):
            if current != label[2:]:  # inconsistance
                if current:
                    spans.append((current, start, i))
                current = label[2:]
                start = i
        else:  # O
            if current:
                spans.append((current, start, i))
            current = None
            start = None
    if current:
        spans.append((current, start, len(labels)))
    return set(spans)

def ner_metrics(y_true, y_pred):
    """Calcule precision, recall, F1 au niveau des entites"""
    spans_true = bio_to_spans(y_true)
    spans_pred = bio_to_spans(y_pred)

    tp = len(spans_true & spans_pred)
    fp = len(spans_pred - spans_true)
    fn = len(spans_true - spans_pred)

    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall    = tp / (tp + fn) if tp + fn > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}

# Exemple de prediction NER
tokens = ["Jean", "Dupont", "travaille", "chez", "BNP", "Paribas", "a", "Paris"]
y_true = ["B-PER", "I-PER",  "O",          "O",    "B-ORG", "I-ORG",    "O", "B-LOC"]
y_pred = ["B-PER", "I-PER",  "O",          "O",    "B-ORG", "I-PER",    "O", "B-LOC"]

print("=== NER: evaluation au niveau entite ===")
print(f"{'Token':12s}  {'True':8s}  {'Pred':8s}  {'Match'}")
for tok, t, p in zip(tokens, y_true, y_pred):
    match = "OK" if t == p else "ERREUR"
    print(f"  {tok:12s}  {t:8s}  {p:8s}  {match}")

metrics = ner_metrics(y_true, y_pred)
print(f"\nMetriques au niveau entite:")
print(f"  True spans : {bio_to_spans(y_true)}")
print(f"  Pred spans : {bio_to_spans(y_pred)}")
print(f"  Precision  : {metrics['precision']:.4f}")
print(f"  Recall     : {metrics['recall']:.4f}")
print(f"  F1         : {metrics['f1']:.4f}")
print(f"  TP={metrics['tp']}, FP={metrics['fp']}, FN={metrics['fn']}")
print("\n  IMPORTANT: evaluation au niveau entite (pas token!)")
print("  => Une entite est correcte SEULEMENT si tous ses tokens sont bons")


In [ ]:
import numpy as np

# ============================================================
# CRF (Conditional Random Field) -- Viterbi decoding
# ============================================================
class SimpleCRF:
    """CRF linaire chaine -- decoding Viterbi"""

    def __init__(self, labels):
        self.labels = labels
        self.n = len(labels)
        self.l2i = {l: i for i, l in enumerate(labels)}
        # Matrice de transition: trans[i, j] = score(label_i -> label_j)
        self.trans = np.random.randn(self.n, self.n) * 0.1
        # Contraintes BIO: I-X ne peut pas suivre B-Y (X != Y) ni O
        self._apply_bio_constraints()

    def _apply_bio_constraints(self):
        """Interdire les transitions invalides (BIO)"""
        for i, l1 in enumerate(self.labels):
            for j, l2 in enumerate(self.labels):
                # I-X doit etre precede de B-X ou I-X
                if l2.startswith("I-"):
                    entity = l2[2:]
                    if l1 not in [f"B-{entity}", f"I-{entity}"]:
                        self.trans[i, j] = -1e9  # interdit

    def viterbi(self, emit_scores):
        """
        emit_scores : (T, n_labels)  scores d emission
        Retourne la meilleure sequence de labels
        """
        T, n = emit_scores.shape
        dp   = np.full((T, n), -np.inf)
        back = np.zeros((T, n), dtype=int)

        dp[0] = emit_scores[0]

        for t in range(1, T):
            for j in range(n):
                scores = dp[t-1] + self.trans[:, j] + emit_scores[t, j]
                best = np.argmax(scores)
                dp[t, j]   = scores[best]
                back[t, j] = best

        # Backtrack
        path = [np.argmax(dp[-1])]
        for t in range(T-1, 0, -1):
            path.append(back[t, path[-1]])
        path.reverse()
        return [self.labels[i] for i in path]

labels = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
crf = SimpleCRF(labels)

# Scores d emission simules (comme la sortie d un BERT)
np.random.seed(42)
tokens = ["Jean", "Dupont", "works", "at", "BNP", "Paribas", "in", "Paris"]
T = len(tokens)
emit = np.random.randn(T, len(labels)) * 0.5

# Forcer des patterns realistes
emit[0, crf.l2i["B-PER"]] += 2.0   # Jean -> B-PER
emit[1, crf.l2i["I-PER"]] += 2.0   # Dupont -> I-PER
emit[4, crf.l2i["B-ORG"]] += 2.0   # BNP -> B-ORG
emit[5, crf.l2i["I-ORG"]] += 2.0   # Paribas -> I-ORG
emit[7, crf.l2i["B-LOC"]] += 2.0   # Paris -> B-LOC

predicted = crf.viterbi(emit)
print("=== CRF Viterbi Decoding ===")
print(f"{'Token':12s}  {'Prediction'}")
for tok, label in zip(tokens, predicted):
    print(f"  {tok:12s}  {label}")
print("\n  => CRF garantit la coherence des sequences BIO")
print("  => BiLSTM-CRF: LSTM fournit les scores d emission, CRF decode")


---
## 7. Language Modeling <a name='lm'></a>

### Objectifs d'entrainement

**Causal Language Modeling (CLM)** — GPT, LLaMA
```
L = -sum_t log P(w_t | w_1, ..., w_{t-1})
```
- Predict le prochain token => bon pour la generation

**Masked Language Modeling (MLM)** — BERT
```
L = -sum_{t in masked} log P(w_t | w_1, ..., w_{t-1}, w_{t+1}, ...)
```
- 15% des tokens masques => apprend des representations contextuelles
- Pas adapte a la generation (voit le futur)

**Next Sentence Prediction (NSP)** — BERT
- Predict si deux phrases sont consecutives
- Abandonne dans RoBERTa (pas utile)

### Perplexite
```
PPL = exp(-1/N * sum log P(w_i | context))
```
- Mesure de qualite d'un LM : plus bas = mieux
- PPL = 10 : le modele a ~10 candidats plausibles a chaque pas
- PPL humain ~ 50-100 sur certains benchmarks

### Loi de scaling (Chinchilla)
- **Chinchilla law** : N_opt ~ 20 * D (N = parametres, D = tokens)
  - 70B params => entrainer sur ~1.4T tokens


In [ ]:
import numpy as np
from collections import defaultdict, Counter

# ============================================================
# N-GRAM LANGUAGE MODEL -- bases theoriques
# ============================================================
class NgramLM:
    """N-gram Language Model avec lissage de Laplace"""

    def __init__(self, n=2, alpha=0.1):
        self.n = n
        self.alpha = alpha   # parametre de lissage
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocab = set()

    def fit(self, corpus):
        for sentence in corpus:
            tokens = ["<s>"] * (self.n - 1) + sentence + ["</s>"]
            self.vocab.update(sentence)
            for i in range(len(tokens) - self.n + 1):
                ngram   = tuple(tokens[i:i+self.n])
                context = tuple(tokens[i:i+self.n-1])
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1
        self.vocab.add("<s>")
        self.vocab.add("</s>")
        self.V = len(self.vocab)
        return self

    def prob(self, word, context):
        """P(word | context) avec lissage de Laplace"""
        ngram = tuple(context) + (word,)
        count_ngram   = self.ngram_counts[ngram]
        count_context = self.context_counts[tuple(context)]
        return (count_ngram + self.alpha) / (count_context + self.alpha * self.V)

    def perplexity(self, sentences):
        """Perplexite sur un corpus"""
        total_log_prob = 0
        total_tokens = 0
        for sentence in sentences:
            tokens = ["<s>"] * (self.n - 1) + sentence + ["</s>"]
            for i in range(self.n - 1, len(tokens)):
                context = tuple(tokens[i-(self.n-1):i])
                word    = tokens[i]
                p = self.prob(word, context)
                total_log_prob += np.log(p + 1e-10)
                total_tokens += 1
        return np.exp(-total_log_prob / total_tokens)

    def generate(self, seed=None, max_len=10):
        context = ["<s>"] * (self.n - 1)
        tokens = []
        for _ in range(max_len):
            candidates = list(self.vocab)
            probs = [self.prob(w, context[-(self.n-1):]) for w in candidates]
            probs = np.array(probs)
            probs /= probs.sum()
            if seed is not None:
                np.random.seed(seed)
            word = np.random.choice(candidates, p=probs)
            if word == "</s>":
                break
            tokens.append(word)
            context.append(word)
        return tokens

# Corpus de demo
corpus = [
    "le chat mange la souris".split(),
    "le chien mange les croquettes".split(),
    "le chat dort sur le canape".split(),
    "la souris mange le fromage".split(),
    "le chien dort dans sa niche".split(),
    "le chat et le chien jouent".split(),
]

print("=== N-gram Language Model ===")
for n in [2, 3]:
    lm = NgramLM(n=n, alpha=0.1).fit(corpus)
    ppl_train = lm.perplexity(corpus)
    print(f"  {n}-gram LM: perplexite train = {ppl_train:.2f}")

bigram = NgramLM(n=2, alpha=0.1).fit(corpus)
print(f"\n  P('mange' | 'chat') = {bigram.prob('mange', ['chat']):.4f}")
print(f"  P('dort'  | 'chat') = {bigram.prob('dort',  ['chat']):.4f}")
print(f"  P('mange' | 'chien') = {bigram.prob('mange', ['chien']):.4f}")
print(f"  P('mange' | 'mange') = {bigram.prob('mange', ['mange']):.4f}  (improbable)")


In [ ]:
import numpy as np

# ============================================================
# MASKED LANGUAGE MODELING (BERT) -- simulation
# ============================================================
def bert_masking(token_ids, vocab_size, mask_token_id=103, mask_prob=0.15, seed=42):
    """
    Applique le masquage BERT:
    - 80% des tokens masques -> [MASK]
    - 10% -> token aleatoire
    - 10% -> token original (sans changement)
    """
    rng = np.random.RandomState(seed)
    input_ids = token_ids.copy()
    labels = np.full_like(token_ids, -100)  # -100 = ignore dans la loss

    for i, token_id in enumerate(token_ids):
        if rng.random() < mask_prob:
            labels[i] = token_id  # on veut predire ce token
            r = rng.random()
            if r < 0.80:
                input_ids[i] = mask_token_id   # remplace par [MASK]
            elif r < 0.90:
                input_ids[i] = rng.randint(0, vocab_size)  # token aleatoire
            # else: garde le token original (10%)

    return input_ids, labels

# Simulation
sentence = [7592, 1010, 2129, 2024, 2017, 1029, 1996, 2308, 2003, 2204]  # IDs BERT
vocab_size = 30522
masked_ids, target_ids = bert_masking(sentence, vocab_size, seed=42)

print("=== BERT Masked Language Modeling ===")
print(f"Original  : {sentence}")
print(f"Masked    : {masked_ids}")
print(f"Labels    : {target_ids}  (-100 = ignore)")
n_masked = (target_ids != -100).sum()
print(f"Tokens masques: {n_masked}/{len(sentence)} ({n_masked/len(sentence)*100:.0f}%)")

# ============================================================
# PERPLEXITE -- interpretation
# ============================================================
print("\n=== Perplexite : interpretation ===")
benchmarks = [
    ("Modele au hasard (|V|=50k)", 50000),
    ("3-gram Wikipedia", 200),
    ("LSTM large", 60),
    ("GPT-2 small", 35),
    ("GPT-2 XL", 18),
    ("GPT-3 175B", 12),
    ("Humain (certains datasets)", 50),
]
for name, ppl in benchmarks:
    log_ppl = np.log(ppl)
    print(f"  {name:35s}: PPL={ppl:6d}  (log={log_ppl:.2f})")
print("\n  PPL = exp(cross-entropy)")
print("  => PPL = 12 signifie: le modele a ~12 tokens plausibles a chaque position")

# ============================================================
# CAUSAL vs MASKED LM
# ============================================================
print("\n=== CLM (GPT) vs MLM (BERT) ===")
print("""
  CLM (Causal):
    - Predict w_t a partir de w_1, ..., w_{t-1}
    - Masque causal (triangulaire inferieur)
    - Adapte a la GENERATION autoregressive
    - Exemples: GPT-2, GPT-3, LLaMA, Mistral

  MLM (Masked):
    - Predict 15% des tokens masques
    - Voit tout le contexte (passe ET futur)
    - Tres bon pour COMPRENDRE (classification, NER, QA)
    - Pas adapte a la generation
    - Exemples: BERT, RoBERTa, DistilBERT

  Combinees (T5, BART):
    - Encoder: MLM ou debruiteur
    - Decoder: CLM
    - Adapte a la traduction, summarisation, QA
""")


---
## 8. Questions d’entretien <a name='questions'></a>

**Q : Pourquoi BPE est-il prefere a la tokenisation par mots ?**
> BPE gere nativement les mots hors-vocabulaire (OOV) en les decomposant en sous-mots connus. Il equilibre la taille du vocabulaire et la longueur des sequences. Un vocabulaire ferme de 32k tokens suffit pour couvrir presque n'importe quelle langue.

**Q : Quelle est la difference entre BERT et GPT ?**
> BERT utilise MLM (voit le contexte des deux cotes) => excellent pour la comprehension. GPT utilise CLM avec masque causal (voit seulement le passe) => adapte a la generation. BERT = encoder only, GPT = decoder only.

**Q : Expliquer l'attention en termes simples.**
> Q (query) = ce que je cherche. K (key) = ce que chaque token offre. V (value) = l'information de chaque token. On calcule un score de compatibilite Q.K^T, on normalise avec softmax, et on fait une somme ponderee des V. Cela permet a chaque token de "regarder" tous les autres.

**Q : Pourquoi diviser par sqrt(d_k) dans l'attention ?**
> Quand d_k est grand, les produits scalaires Q.K^T deviennent grands en norme, poussant le softmax dans des regions a gradient tres faible. Diviser par sqrt(d_k) recentre les scores => gradients stables.

**Q : Qu'est-ce que la perplexite et comment l'interpreter ?**
> PPL = exp(cross-entropie). C'est le nombre moyen de tokens plausibles au yeux du modele a chaque position. PPL=10 => modele tres confiant. PPL=50000 => modele aleatoire. On compare toujours sur le meme tokeniseur et le meme dataset.

**Q : BERT vs RoBERTa : quelles differences ?**
> RoBERTa : (1) supprime le NSP (inutile), (2) batch plus grand, (3) sequences plus longues, (4) plus de donnees, (5) masquage dynamique. Resultat : RoBERTa > BERT sur la plupart des benchmarks.

**Q : Qu'est-ce que le NER et comment l'evaluer correctement ?**
> NER = identifier et classer des entites nommees (PER, ORG, LOC...). L'evaluation se fait au niveau entite (span-level), pas au niveau token : une entite est correcte seulement si TOUS ses tokens et son type sont corrects. Utiliser seqeval en pratique.

**Q : Quelle est la complexite de l'attention ?**
> O(n^2 * d) en temps et espace, ou n est la longueur de la sequence. C'est le goulot d'etranglement pour les longues sequences. Solutions : Sparse attention (Longformer), Linear attention, Flash Attention (IO-aware).
